In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## import

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [4]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention


In [5]:
import scanpy as sc
import spatialdata as sd
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)


In [6]:
import Multi_Modal_Hadmard as MMR

/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [7]:
import importlib
import Multi_Modal_Hadmard.tools
import Multi_Modal_Hadmard.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

In [ ]:
adata = sdata.tables["table"]
adata

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

## Train

In [18]:
adata = sc.read_h5ad("../../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

Founsation Models

In [19]:
rng = np.random.default_rng(42)

edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal(M.shape)

In [21]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [9]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [9]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [10]:
edata.obs_names = [cid[61:] for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['h_optimus'][edata.obs_names.get_indexer(common_cells)]

In [20]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=500)
X_reduced = pca.fit_transform(adata.obsm['Morpho_Embedding'])
adata.obsm['p_Morpho_Embedding'] = X_reduced

Preparing Dataset

In [21]:
adata = MMR.prep_adatas(adata, norm=True, log1p=True)
dataset = MMR.make_dataset(adata, sparse_graph=True)

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.


In [22]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)

Expression torch.Size([63173, 2000])
Morpho_Embedding torch.Size([63173, 1536])
Neighborhood_Graph torch.Size([2, 505384])


In [41]:
importlib.reload(MMR.dataset)
importlib.reload(MMR.model)
importlib.reload(MMR)

<module 'Multi_Modal_Hadmard' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/Hadmard_Attention/../../Multi_Modal_Hadmard/__init__.py'>

In [17]:
import gc

gc.collect()
torch.cuda.empty_cache()

# Train

## Model Type 0

In [14]:
model_type = 0

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [15]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10489
Epoch 101: loss =  0.08975
Epoch 201: loss =  0.08199
Epoch 301: loss =  0.08180
Epoch 401: loss =  0.08178
Epoch 501: loss =  0.08177
Epoch 601: loss =  0.08177
Stopping criterion met. Final loss =  0.08177


Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [15]:
#H_OPTIMUS
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10488
Epoch 101: loss =  0.08976
Epoch 201: loss =  0.08202
Epoch 301: loss =  0.08180
Epoch 401: loss =  0.08178
Epoch 501: loss =  0.08177
Epoch 601: loss =  0.08177
Stopping criterion met. Final loss =  0.08177


Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [15]:
#VIRCHOW
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  0
Epoch 1: loss =  0.10488
Epoch 101: loss =  0.08929
Epoch 201: loss =  0.08192
Epoch 301: loss =  0.08180
Epoch 401: loss =  0.08178
Epoch 501: loss =  0.08177
Epoch 601: loss =  0.08177
Stopping criterion met. Final loss =  0.08177


Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2560, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [26]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
Epoch 1: loss =  0.10296
Epoch 101: loss =  0.08877
Epoch 201: loss =  0.08190
Epoch 301: loss =  0.08179
Epoch 401: loss =  0.08111
Epoch 501: loss =  0.07059
Epoch 601: loss =  0.06293
Epoch 701: loss =  0.05987
Epoch 801: loss =  0.05881
Epoch 901: loss =  0.05848
Epoch 1001: loss =  0.05823
Epoch 1101: loss =  0.05738
Epoch 1201: loss =  0.05690
Epoch 1301: loss =  0.05665
Epoch 1401: loss =  0.05658
Epoch 1501: loss =  0.05643
Epoch 1601: loss =  0.05628
Epoch 1701: loss =  0.05595
Epoch 1801: loss =  0.05558
Epoch 1901: loss =  0.05549
Epoch 2001: loss =  0.05541
Epoch 2101: loss =  0.05535
Epoch 2201: loss =  0.05532
Epoch 2301: loss =  0.05532
Epoch 2401: loss =  0.05531
Epoch 2501: loss =  0.05528
Epoch 2601: loss =  0.05525
Epoch 2701: loss =  0.05525
Epoch 2801: loss =  0.05525
Epoch 2901: loss =  0.05523
Epoch 3001: loss =  0.05519
Epoch 3101: loss =  0.05521
Epoch 3201: loss =  0.05517
Epoch 3301: loss =  0.05511
Epoch 3401: loss =  0.05508
Epoch 35

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [27]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_UNI_{model_type}.pth',weights_only=True))

## Model Type 1

In [ ]:
model_type = 1

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.10954, morpho_loss =  0.19044

## Model Type 2

In [ ]:
model_type = 2

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.10763, morpho_loss =  0.18602

## Model Type 3

In [ ]:
model_type = 3

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.10801, morpho_loss =  0.19012

## Model Type 4

In [ ]:
model_type = 4

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [ ]:
model.fit(dataset, entry_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/Xbcancer_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/Xbcancer_UNI_{model_type}.pth',weights_only=True))

Final Loss: gene_loss =  0.11449, morpho_loss =  0.18615

## Model Type 5

In [23]:
model_type = 5

MMR.set_random_seed(0)
model = MMR.model.Steamboat(features=len(adata.var_names.tolist()), morpho_features=adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, model_type=model_type, n_scales=2)
model = model.to(device)


In [24]:
#NOISE
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10488
Epoch 101: loss =  0.08724
Epoch 201: loss =  0.06754
Epoch 301: loss =  0.06257
Epoch 401: loss =  0.06040
Epoch 501: loss =  0.05908
Epoch 601: loss =  0.05879
Epoch 701: loss =  0.05867
Epoch 801: loss =  0.05857
Epoch 901: loss =  0.05850
Epoch 1001: loss =  0.05831
Epoch 1101: loss =  0.05806
Epoch 1201: loss =  0.05792
Epoch 1301: loss =  0.05786
Epoch 1401: loss =  0.05782
Epoch 1501: loss =  0.05778
Epoch 1601: loss =  0.05776
Epoch 1701: loss =  0.05749
Epoch 1801: loss =  0.05725
Epoch 1901: loss =  0.05716
Epoch 2001: loss =  0.05711
Epoch 2101: loss =  0.05710
Epoch 2201: loss =  0.05708
Epoch 2301: loss =  0.05708
Epoch 2401: loss =  0.05705
Epoch 2501: loss =  0.05705
Epoch 2601: loss =  0.05703
Epoch 2701: loss =  0.05703
Epoch 2801: loss =  0.05702
Epoch 2901: loss =  0.05701
Epoch 3001: loss =  0.05702
Epoch 3101: loss =  0.05701
Epoch 3201: loss =  0.05699
Epoch 3301: loss =  0.05698
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [17]:
#H_OPTIMUS
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10488
Epoch 101: loss =  0.08723
Epoch 201: loss =  0.06750
Epoch 301: loss =  0.06248
Epoch 401: loss =  0.06085
Epoch 501: loss =  0.06020
Epoch 601: loss =  0.05999
Epoch 701: loss =  0.05988
Epoch 801: loss =  0.05979
Epoch 901: loss =  0.05965
Epoch 1001: loss =  0.05912
Epoch 1101: loss =  0.05892
Epoch 1201: loss =  0.05860
Epoch 1301: loss =  0.05845
Epoch 1401: loss =  0.05832
Epoch 1501: loss =  0.05799
Epoch 1601: loss =  0.05733
Epoch 1701: loss =  0.05724
Epoch 1801: loss =  0.05719
Epoch 1901: loss =  0.05715
Epoch 2001: loss =  0.05712
Epoch 2101: loss =  0.05708
Epoch 2201: loss =  0.05692
Epoch 2301: loss =  0.05655
Epoch 2401: loss =  0.05648
Epoch 2501: loss =  0.05646
Epoch 2601: loss =  0.05643
Epoch 2701: loss =  0.05641
Epoch 2801: loss =  0.05640
Epoch 2901: loss =  0.05635
Epoch 3001: loss =  0.05620
Epoch 3101: loss =  0.05608
Epoch 3201: loss =  0.05601
Epoch 3301: loss =  0.05600
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [29]:
#UNI
model.fit(dataset, entry_masking_rate=0.8,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=100, stop_tol=200)

This is the new version
model_type:  5
Epoch 1: loss =  0.10488
Epoch 101: loss =  0.08723
Epoch 201: loss =  0.06758
Epoch 301: loss =  0.06262
Epoch 401: loss =  0.06100
Epoch 501: loss =  0.05984
Epoch 601: loss =  0.05945
Epoch 701: loss =  0.05926
Epoch 801: loss =  0.05873
Epoch 901: loss =  0.05819
Epoch 1001: loss =  0.05803
Epoch 1101: loss =  0.05797
Epoch 1201: loss =  0.05790
Epoch 1301: loss =  0.05779
Epoch 1401: loss =  0.05768
Epoch 1501: loss =  0.05748
Epoch 1601: loss =  0.05730
Epoch 1701: loss =  0.05720
Epoch 1801: loss =  0.05715
Epoch 1901: loss =  0.05711
Epoch 2001: loss =  0.05708
Epoch 2101: loss =  0.05707
Epoch 2201: loss =  0.05705
Epoch 2301: loss =  0.05705
Epoch 2401: loss =  0.05701
Epoch 2501: loss =  0.05701
Epoch 2601: loss =  0.05654
Epoch 2701: loss =  0.05646
Epoch 2801: loss =  0.05643
Epoch 2901: loss =  0.05642
Epoch 3001: loss =  0.05642
Epoch 3101: loss =  0.05626
Epoch 3201: loss =  0.05615
Epoch 3301: loss =  0.05612
Epoch 3401: loss =  0

Steamboat(
  (spatial_gather): HadmardAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (encoder_g): Encoder(
      (network): Sequential(
        (0): Linear(in_features=2000, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=1024, bias=True)
        (3): ReLU()
        (4): Linear(in_features=1024, out_features=2000, bias=True)
        (5): ReLU()
      )
    )
    (q_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (q_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_m): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_g): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local_m): NonNegLinear(
      (elu): ELU(a

In [27]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), f'saved_models/mouse_brain_UNI_{model_type}.pth')
else:
    model.load_state_dict(torch.load(f'saved_models/mouse_brain_UNI_{model_type}.pth',weights_only=True))